# Prompting, Decoding, and Confident Wrongness

**Computational Text Analysis Pack · Module 27 — Generative Models in DH Research**

*Technique:* Decoding strategies; hallucination on African literary material

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/14-generative-llms/27a_prompting_and_hallucination.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

A generative model produces fluent text about anything you ask. Fluency is not
accuracy, and the gap between them is widest exactly where this course works:
African literature, African languages, and anything outside the English-language
canon that dominates pretraining data.

This notebook makes that gap measurable rather than anecdotal.

## The tooling

**HuggingFace `transformers`** runs open models locally, including the decoding
strategies below. **vLLM** and **llama.cpp** are the usual serving layers when
you need throughput. Hosted APIs — OpenAI, Anthropic, Cohere — are what most
researchers actually reach for.

In CourseHub these cells go through a **gateway** that holds the credential
server-side and routes to open models. In Colab you supply your own key, or the
notebook falls back to recorded output. No API key appears in any cell of this
course, because a notebook cell is readable by whoever runs it.

**The cost question is a methods question.** Generating over a 5,000-document
corpus is not free and not fast, and unlike a frequency count it is not
deterministic — two runs give two answers. Budget and reproducibility both bite.

In [ ]:
import sys, os, json, textwrap

IN_COLAB = "google.colab" in sys.modules
LLM_BACKEND = "recorded"
_chat = None

if not IN_COLAB:
    try:
        sys.path.insert(0, "/app")
        from utils.llm_gateway import gateway_chat as _gw
        def _chat(messages, model="gpt-oss-20b", max_tokens=300):
            # allow_fallback=False keeps us on the local provider; the fallback
            # is billed per token and a classroom can run a lot of tokens.
            txt, meta = _gw(messages, model=model, max_tokens=max_tokens,
                            allow_fallback=False)
            return txt, meta.get("served_model", model)
        _chat([{"role": "user", "content": "ok"}], max_tokens=5)
        LLM_BACKEND = "coursehub-gateway"
    except Exception as exc:
        _why = str(exc)[:80]
else:
    key = os.environ.get("OPENAI_API_KEY", "")
    if key:
        def _chat(messages, model="gpt-4o-mini", max_tokens=300):
            import requests
            r = requests.post("https://api.openai.com/v1/chat/completions",
                              headers={"Authorization": f"Bearer {key}"},
                              json={"model": model, "messages": messages,
                                    "max_tokens": max_tokens}, timeout=60)
            r.raise_for_status()
            return r.json()["choices"][0]["message"]["content"], model
        LLM_BACKEND = "colab-openai"
    else:
        _why = "no OPENAI_API_KEY set in this Colab session"

def ask(prompt, model=None, max_tokens=300, system=None):
    """One prompt in, one string out — or a recorded answer when offline."""
    if _chat is None:
        return None, "recorded"
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    kw = {"max_tokens": max_tokens}
    if model:
        kw["model"] = model
    return _chat(msgs, **kw)

print(f"LLM backend: {LLM_BACKEND}")
if LLM_BACKEND == "recorded":
    print(f"   ({_why})")
    print("   Cells below print outputs recorded when this notebook was written,")
    print("   clearly marked, so the lesson still works without a model.")
elif LLM_BACKEND == "colab-openai":
    print("   using your own key from this Colab session")
else:
    print("   using the CourseHub gateway; no key needed and none is exposed here")

## Decoding: the setting that changes the text more than the model does

Before prompting, understand the knob. A model outputs a probability
distribution over the next token; **decoding** is how you pick from it.

- **Greedy** — always the highest-probability token. Deterministic, repetitive.
- **Beam search** — keep the *n* best sequences. Good for translation, bland for prose.
- **Top-k / nucleus (top-p)** — sample from the most probable tokens only.
  Fluent and varied, and the source of most hallucination.
- **Temperature** — flattens or sharpens the distribution before sampling.

For research the important property is that sampling is **not reproducible**.
Same prompt, same model, different answer.

In [ ]:
# Decoding on a small local model, so the mechanism is visible and free.
import numpy as np

probs = np.array([0.40, 0.25, 0.15, 0.10, 0.05, 0.03, 0.02])
tokens = ["the", "a", "his", "her", "their", "one", "some"]

def temper(p, t):
    logits = np.log(np.clip(p, 1e-12, None)) / t
    e = np.exp(logits - logits.max())
    return e / e.sum()

print(f"{'token':<10}{'raw':>8}{'T=0.5':>9}{'T=1.0':>9}{'T=1.5':>9}")
print("-" * 46)
for i, tk in enumerate(tokens):
    print(f"{tk:<10}{probs[i]:>8.3f}{temper(probs,0.5)[i]:>9.3f}"
          f"{temper(probs,1.0)[i]:>9.3f}{temper(probs,1.5)[i]:>9.3f}")
print()
print("Low temperature concentrates on 'the'; high temperature spreads out and")
print("lets unlikely tokens through. Nothing about the model changed.")

fig, ax = plt.subplots(figsize=(7, 3.6))
w = 0.25
x = np.arange(len(tokens))
for j, t in enumerate([0.5, 1.0, 1.5]):
    ax.bar(x + (j - 1) * w, temper(probs, t), width=w, label=f"T={t}")
ax.set_xticks(x); ax.set_xticklabels(tokens); ax.legend()
ax.set_ylabel("probability"); ax.set_title("Temperature reshapes the distribution")
plt.tight_layout()

## The test that matters for this course

Ask about a canonical African novel. *The Palm-Wine Drinkard* (1952) is by
**Amos Tutuola** — one of the seven authors in the case study in module 13, and
one of the most studied African novels in English.

In [ ]:
FACT_PROBES = [
    ("Who wrote the novel 'The Palm-Wine Drinkard', and in what year was it "
     "first published? Answer in one sentence.",
     "Amos Tutuola, 1952"),
    ("Who wrote 'Things Fall Apart', and in what year? One sentence.",
     "Chinua Achebe, 1958"),
    ("Who wrote 'The Joys of Motherhood', and in what year? One sentence.",
     "Buchi Emecheta, 1979"),
]
if LLM_BACKEND != "recorded":
    for prompt, truth in FACT_PROBES:
        out, served = ask(prompt, max_tokens=90)
        print(f"Q: {prompt.split('?')[0]}?")
        print(f"   model  : {' '.join(out.split())[:150]}")
        print(f"   truth  : {truth}")
        print()
else:
    print("RECORDED (gpt-oss-20b, via the gateway, when this notebook was written):\n")
    print("Q: Who wrote 'The Palm-Wine Drinkard'?")
    print("   model  : The novel **'The Palm-Wine Drinkard' was written by Chinua")
    print("            Achebe**. It was first published in **1953**.")
    print("   truth  : Amos Tutuola, 1952")
print()
print("Read the first answer carefully. On the run recorded above the model")
print("attributed Tutuola's novel to Achebe, gave the wrong year, and set both")
print("in bold. Confidence is a property of the surface, not of the knowledge.")

## Fabrication, not just error

Getting a fact wrong is one failure. Inventing a source is a worse one, because
there is nothing to check it against — and it is the failure that ends careers
when it reaches a bibliography.

The cell below asks about a novel that **does not exist**. The title is a
distortion of Fágúnwà's *Ìrìnkèrindò nínú Igbó Elégbèje*, attributed to a real
but different Yorùbá writer.

In [ ]:
FAKE = ("In two sentences, summarise the plot of the novel "
        "'Irinkerindo Oloro' by the Yoruba writer Adebayo Faleti.")
if LLM_BACKEND != "recorded":
    out, served = ask(FAKE, max_tokens=150)
    print("prompt asked about a NON-EXISTENT novel.\n")
    print("model:", " ".join(out.split())[:340])
else:
    print("RECORDED: the model produced a confident plot summary —")
    print("  'follows the surreal journey of a wandering sage who encounters")
    print("   bizarre, allegorical characters that mirror the political and")
    print("   cultural unrest of contemporary Yoruba society...'")
print()
print("No hedge, no 'I am not aware of this work'. The summary is plausible")
print("because it is assembled from the shape of Yoruba literary criticism the")
print("model HAS seen. That is what makes it dangerous: it is wrong in a")
print("register that reads exactly like scholarship.")

## What to take from this

1. **Never use a generative model as a source of fact** about a literature it may
   barely have seen. Use it on text you supply — which is the next notebook.
2. **Ask for citations and then check them.** An invented citation is the
   commonest and most damaging failure.
3. **Sampling means irreproducibility.** Record the model, the version, the
   temperature and the date, or your result cannot be rerun — not even by you.
4. **The failure is uneven.** These models are far more reliable on Achebe than
   on Fálétí, and more reliable in English than in Yorùbá. Your error rate
   depends on whose literature you study.

### 🔧 Your turn 1

Probe the model on three works central to your own research, where you already know the answer. What is your error rate?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Researchers working on canonical anglophone material often report near-zero
errors and conclude the tool is reliable; researchers working on African-language
or small-press material frequently find errors in the majority of answers. Both
are measuring the same model. That divergence is the finding, and it means a
colleague's experience of "it works fine" carries no information about your
material. Test on your own corpus before trusting it on your own corpus.

</details>